<a href="https://colab.research.google.com/github/google-deepmind/hybrid_rnns_reward_learning/blob/main/hybrid_rnns_reward_learning/train_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs, imports, etc.

In [1]:
#@title Install Hybrid RNNs repo from github
# !rm -r hybrid_rnns_reward_learning
#!git clone https://github.com/google-deepmind/hybrid_rnns_reward_learning
#%cd hybrid_rnns_reward_learning
#!pip install .
#%cd ..

In [1]:
#@title Import hybRNN functions
from hybrid_rnns_reward_learning import bi_rnn
from hybrid_rnns_reward_learning import cogmod
from hybrid_rnns_reward_learning import fit_hyb_rnn
from hybrid_rnns_reward_learning import hyb_rnn_utilities
from hybrid_rnns_reward_learning import rnn
from hybrid_rnns_reward_learning import rnn_config

In [2]:
#@title Download human dataset from OSF
import urllib
import os
import pandas as pd
import haiku as hk

download_url = 'https://osf.io/download/dw7f6/'
local_path = os.path.join('/tmp', 'openSourceRawDataset.csv')
urllib.request.urlretrieve(download_url, local_path)

data_df = pd.read_csv(local_path)
print(data_df.head())

   s_id  block  trial_id  action  reward     rt  payout_1  payout_2  payout_3  \
0   268    1.0         0     0.0    0.75  379.0      0.75      0.97      0.26   
1   268    1.0         1     2.0    0.28  212.0      0.74      0.99      0.28   
2   268    1.0         2     1.0    0.93  128.0      0.72      0.93      0.34   
3   268    1.0         3     3.0    0.47   68.0      0.73      0.98      0.22   
4   268    1.0         4     1.0    0.96  272.0      0.65      0.96      0.25   

   payout_4  
0      0.59  
1      0.53  
2      0.49  
3      0.47  
4      0.55  


# Set up training config

In [3]:
#@title Example 1: Fit "Best RL" model (free parameters: α, β, b, κ, Qinit, p)
config = rnn_config.get_config()
config.model_name = 'cogmod'
config.rnn_rl_params.fit_alpha = True
config.rnn_rl_params.fit_beta = True
config.rnn_rl_params.fit_bias = True
config.rnn_rl_params.fit_forget = True
config.rnn_rl_params.fit_init_v = True
config.rnn_rl_params.fit_persev_t = True
config.rnn_rl_params.fit_init_h = False
config.rnn_rl_params.fit_persev_p = False
config.rnn_rl_params.fit_w = False

In [4]:
#@title Example 2: Fit winning "hybRNN" model
config = rnn_config.get_config()
config.model_name = 'birnn'
config.rnn_rl_params.w_v = 1
config.rnn_rl_params.w_h = 1
config.rnn_rl_params.fit_forget = True
config.rnn_rl_params.o = False
config.rnn_rl_params.s = True
config.rnn_rl_params.zero_values = True
config.rnn_rl_params.fit_init_v = True
config.rnn_rl_params.fit_init_h = True

In [9]:
config.dataset_path = local_path
config.n_training_steps = 1001 #base: 1001
config.batch_size = 32
# config

# Run the training loop

In [10]:
scalars, params = fit_hyb_rnn.train(config)

Loading data from /tmp/openSourceRawDataset.csv
Size of training data: 415 blocks.
Using BiRNN to fit data.
Start fitting the model
Step: 0,
Scalars: {'train_loss': [array(360.2608, dtype=float32)], 'step': [0], 'test_loss': [array(338.92624, dtype=float32)], 'valid_loss': [array(329.8441, dtype=float32)]}
Step: 500,
Scalars: {'train_loss': [array(133.59103, dtype=float32)], 'step': [500], 'test_loss': [array(139.77043, dtype=float32)], 'valid_loss': [array(141.23744, dtype=float32)]}
Step: 1000,
Scalars: {'train_loss': [array(90.09355, dtype=float32)], 'step': [1000], 'test_loss': [array(120.307785, dtype=float32)], 'valid_loss': [array(112.53139, dtype=float32)]}


In [11]:
#@title Inspect the loss
scalars

{'train_loss': [array(90.09355, dtype=float32)],
 'step': [1000],
 'test_loss': [array(120.307785, dtype=float32)],
 'valid_loss': [array(112.53139, dtype=float32)]}

In [12]:
#@title Inspect fitted model parameters
params

{'bi_rnn': {'init_value_h': Array([0.6355823], dtype=float32),
  'init_value_v': Array([1.3579535], dtype=float32),
  'unsigmoid_forget': Array([-0.80390024], dtype=float32)},
 'bi_rnn/~_habit_rnn/linear': {'b': Array([ 0.05596803, -0.04695085,  0.01295873,  0.02051177,  0.01868691,
          0.02152276, -0.05629874,  0.04196452,  0.01869367,  0.02671932,
          0.00417171, -0.03265128, -0.04571581,  0.00919976,  0.00647139,
         -0.03222556], dtype=float32),
  'w': Array([[-0.06704309,  0.07041682,  0.05734549,  0.46440572, -0.14880322,
          -0.10997928, -0.06818479,  0.13355897, -0.20150134,  0.08573952,
          -0.32597998, -0.02477754, -0.30126312,  0.09226436,  0.0121372 ,
          -0.18184946],
         [-0.36887997,  0.25869834, -0.02198926, -0.14670971, -0.20991243,
           0.07638509,  0.15049244, -0.3038613 ,  0.02720435,  0.10694532,
           0.02389707, -0.14918283,  0.08385012,  0.10334948,  0.00907495,
           0.22469978],
         [-0.00218786, -0.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# 1. Real data check
print("Shape:", data_df.shape)
print("Subjects:", data_df['s_id'].nunique())
print("Blocks per subject (mean):", data_df.groupby('s_id')['block'].nunique().mean())
print("Columns:", list(data_df.columns))

# 2. Loss curve — only 3 logged points (step 0, 500, 1000), so plot as scatter
print("Scalars keys:", list(scalars.keys()))
plt.plot(scalars['step'], scalars['train_loss'], 'o-', label='train')
plt.plot(scalars['step'], scalars['valid_loss'], 'o-', label='valid')
plt.plot(scalars['step'], scalars['test_loss'],  'o-', label='test')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.show()

# 3. Parameter count
n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"{n_params:,} parameters")